In [1]:
!pip install transformers datasets accelerate -q


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

from sklearn.metrics import accuracy_score, f1_score

In [3]:
train_df = pd.read_csv("../data/processed/train_processed.csv")
valid_df = pd.read_csv("../data/processed/valid_processed.csv")

print(train_df.shape)
print(valid_df.shape)

(9539, 3)
(2388, 3)


In [4]:
train_df = train_df.sample(2000, random_state=42)
valid_df = valid_df.sample(500, random_state=42)

In [5]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tanis\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [6]:


def tokenize_function(example):

    return tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["clean_text", "label"]]
)

In [8]:
train_dataset = train_dataset.map(tokenize_function)

valid_dataset = valid_dataset.map(tokenize_function)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

In [10]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [11]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [14]:
training_args = TrainingArguments(

    output_dir="../outputs/finbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_dir="../outputs/logs",

    load_best_model_at_end=True,

    metric_for_best_model="f1"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.628872,0.754000,0.718266
2,0.625760,0.668087,0.778000,0.773045
3,0.625760,0.748196,0.778000,0.776972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.5242887369791667, metrics={'train_runtime': 3912.0538, 'train_samples_per_second': 1.534, 'train_steps_per_second': 0.192, 'total_flos': 394670126592000.0, 'train_loss': 0.5242887369791667, 'epoch': 3.0})

In [17]:
trainer.save_model("../models/bert/finbert_model")

tokenizer.save_pretrained("../models/bert/finbert_tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/bert/finbert_tokenizer\\tokenizer_config.json',
 '../models/bert/finbert_tokenizer\\tokenizer.json')

In [ ]:
import torch

label_map = {
    0: "Bearish",
    1: "Neutral",
    2: "Bullish"
}

def predict_finbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.nn.functional.softmax(
            outputs.logits,
            dim=-1
        )

        prediction = torch.argmax(probs).item()

        confidence = torch.max(probs).item()

    return {
        "sentiment": label_map[prediction],
        "confidence": round(confidence, 4)
    }

In [19]:
predict_finbert(
    "Tesla stock surged after record quarterly earnings"
)

{'sentiment': 'Neutral', 'confidence': 0.6771}

In [20]:
predict_finbert(
    "The company suffered massive financial losses"
)

{'sentiment': 'Negative', 'confidence': 0.838}

In [21]:
predict_finbert(
    "The board announced its annual meeting"
)

{'sentiment': 'Positive', 'confidence': 0.992}

# FinBERT Evaluation Summary

## Observations

- FinBERT achieved competitive performance on financial sentiment classification.
- Transformer-based contextual embeddings improved semantic understanding.
- Limited training data reduced generalization capability.
- FinBERT required significantly higher computational resources compared to RNN, LSTM, and GRU.

## Advantages

- Better contextual understanding
- Handles complex sentence structures
- Pretrained on financial text corpus
- Strong performance on domain-specific NLP tasks

## Limitations

- Longer training time
- Higher memory usage
- Requires larger datasets for optimal fine-tuning
- Sensitive to class imbalance

## Conclusion

FinBERT demonstrated strong potential for financial sentiment analysis. While GRU and LSTM achieved comparable accuracy on smaller datasets, transformer-based architectures are more scalable and effective for large-scale NLP applications.